# Chroma Backend — Lightweight Local Vector Store

Medha 0.3.1 ships with `ChromaBackend`: a storage backend that uses
[Chroma](https://www.trychroma.com/) for vector search. Chroma offers three
operating modes, making it ideal for prototyping, CI pipelines, and lightweight
deployments that don't want to run a dedicated vector database.

## Three Modes

| Mode | `chroma_mode` | Data location | Use case |
|---|---|---|---|
| **Ephemeral** | `"ephemeral"` | In-process RAM | Tests, demos, CI |
| **Persistent** | `"persistent"` | Local filesystem | Single-node, survives restarts |
| **HTTP** | `"http"` | Remote Chroma server | Multi-process, shared |

This notebook covers:
1. **Ephemeral mode** — zero config, zero cleanup
2. **Persistent mode** — local disk, survives restarts
3. **HTTP mode** — remote Chroma server
4. **Authentication** — bearer token for protected instances
5. **When to choose Chroma**

**Requirements:**
```bash
pip install "medha-archai[chroma,fastembed]"
```

## Prerequisites

```bash
pip install "medha-archai[chroma,fastembed]"
```

Ephemeral and persistent modes need **no Docker** — Chroma runs in-process.

For HTTP mode:
```bash
# Start a Chroma server
pip install chromadb
chroma run --host 0.0.0.0 --port 8000
```

In [ ]:
import os
import shutil

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

try:
    from medha import ChromaBackend
    HAS_CHROMA = True
    print("chromadb package is installed")
except ImportError:
    HAS_CHROMA = False
    print("chromadb not found — install with: pip install medha-archai[chroma]")

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

PERSIST_PATH = "/tmp/medha_chroma_demo"

## Cell 1: Ephemeral Mode — Zero Config, Zero Cleanup

In [ ]:
if not HAS_CHROMA:
    print("Skipping — chromadb not installed.")
else:
    settings = Settings(
        backend_type="chroma",
        chroma_mode="ephemeral",
    )

    pairs = [
        ("How many users are there?",     "SELECT COUNT(*) FROM users"),
        ("List active users",             "SELECT * FROM users WHERE status = 'active'"),
        ("Average order value",           "SELECT AVG(total) FROM orders"),
    ]

    async with Medha("chroma_ephemeral", embedder=embedder, settings=settings) as medha:
        for q, sql in pairs:
            await medha.store(q, sql)

        hit = await medha.search("Total number of users")
        print(f"strategy   : {hit.strategy.value}")
        print(f"confidence : {hit.confidence:.4f}")
        print(f"query      : {hit.generated_query}")

    print("\nEphemeral session closed — data is gone.")

## Cell 2: Persistent Mode — Survives Restarts

Data is written to `chroma_persist_path` on disk. Close and reopen Medha
with the same path and the entries are still there.

In [ ]:
if not HAS_CHROMA:
    print("Skipping — chromadb not installed.")
else:
    # Clean slate
    shutil.rmtree(PERSIST_PATH, ignore_errors=True)

    settings = Settings(
        backend_type="chroma",
        chroma_mode="persistent",
        chroma_persist_path=PERSIST_PATH,
    )

    # --- Session 1: store data ---
    print("Session 1: storing entries...")
    async with Medha("chroma_persist", embedder=embedder, settings=settings) as medha:
        await medha.store("Total revenue this month",
                          "SELECT SUM(amount) FROM orders WHERE MONTH(created_at) = MONTH(NOW())")
        await medha.store("Active users count",
                          "SELECT COUNT(*) FROM users WHERE status = 'active'")
        count = await medha._backend.count("chroma_persist")
        print(f"  Stored 2 entries (count={count})")

    # --- Session 2: reopen — data persists ---
    print("\nSession 2: reopening with same persist path...")
    async with Medha("chroma_persist", embedder=embedder, settings=settings) as medha:
        count = await medha._backend.count("chroma_persist")
        print(f"  Entries after reopen: {count}")
        hit = await medha.search("How much revenue did we make this month?")
        print(f"  strategy={hit.strategy.value}  query={hit.generated_query!r}")

## Cell 3: HTTP Mode — Remote Chroma Server

HTTP mode connects to a running Chroma server. Multiple processes can share
the same Chroma instance.

Start a local Chroma server first:
```bash
chroma run --host 0.0.0.0 --port 8000
```

In [ ]:
import urllib.request
try:
    urllib.request.urlopen("http://localhost:8000", timeout=2)
    CHROMA_HTTP_UP = True
except Exception:
    CHROMA_HTTP_UP = False

if not HAS_CHROMA or not CHROMA_HTTP_UP:
    print("Skipping — Chroma HTTP server not running.")
    print("Start with: chroma run --host 0.0.0.0 --port 8000")
else:
    settings = Settings(
        backend_type="chroma",
        chroma_mode="http",
        chroma_host="localhost",
        chroma_port=8000,
    )

    async with Medha("chroma_http", embedder=embedder, settings=settings) as medha:
        await medha.store("Show all products", "SELECT * FROM products")
        hit = await medha.search("List all products")
        print(f"strategy={hit.strategy.value}  query={hit.generated_query!r}")

## Cell 4: Authentication — Bearer Token for Protected Instances

In [ ]:
# Token authentication (for Chroma HTTP server with auth enabled)
# This cell demonstrates the settings — the actual server is not required.
settings_auth = Settings(
    backend_type="chroma",
    chroma_mode="http",
    chroma_host="my-chroma-server.example.com",
    chroma_port=8000,
    chroma_ssl=True,
    chroma_auth_token="my-bearer-token",  # MEDHA_CHROMA_AUTH_TOKEN
)
print(f"auth_token set : {settings_auth.chroma_auth_token is not None}")
print(f"ssl            : {settings_auth.chroma_ssl}")
print(f"host           : {settings_auth.chroma_host}:{settings_auth.chroma_port}")

## When to Choose Chroma

| Situation | Recommendation |
|---|---|
| Prototyping, no infra | `chroma` ephemeral — zero setup |
| CI/CD pipeline | `memory` (fastest) or `chroma` ephemeral |
| Single-node with persistence | `chroma` persistent — simple, no server |
| Multi-process shared cache | `chroma` http or `qdrant` docker |
| Production, high load | `qdrant` or `pgvector` — better scalability |

**Best fit:** Chroma is ideal for **prototypes, local development, and CI** where
you want persistence without running a dedicated server. For production workloads
with high concurrency, prefer `qdrant` or `pgvector`.